# Inspect swing-shape clustering

- `swings` - competitive-swing model table (`data/swings_model.parquet`)
- `assign` - per-swing cluster assignment (`play_id, batter_id, cluster, resp_max`)
- `summary` - per-(batter, cluster) raw-unit centroids + usage weight
- `rep` - per-batter repertoire (`k, bic, effective_shapes`, ...)
- `swc` - `swings` left-joined with `assign` (per-swing frame with features + context + value + cluster)

Helpers: `SHAPE` (5 clustering features), `RAW_SHAPE` (unmirrored), `by_name(s)` to find a batter_id.

In [ ]:
library(arrow)
library(tidyverse)
library(ggthemes)
library(IRdisplay)

options(repr.plot.width = 14, repr.plot.height = 6, repr.plot.res = 150)

ROOT  <- if (dir.exists(file.path(getwd(), "data"))) getwd() else dirname(getwd())
DATA  <- file.path(ROOT, "data")
PLOTS <- file.path(ROOT, "results", "plots")
dir.create(PLOTS, recursive = TRUE, showWarnings = FALSE)

FT_THEME <- theme_fivethirtyeight() +
  theme(axis.title        = element_text(),
        axis.text         = element_text(size = 10),
        panel.background  = element_rect(fill = "white", color = NA),
        plot.background   = element_rect(fill = "white", color = NA),
        legend.background = element_rect(fill = "white", color = NA),
        strip.text        = element_text(size = 12, face = "bold"),
        plot.title        = element_text(size = 16, face = "bold"))

plot_path <- function(name) {
  sub <- if (grepl("^swing_cards",   name)) "swing_cards"
    else if (grepl("^usage_heatmap", name)) "usage_heatmap"
    else if (grepl("^repertoire",    name)) "repertoire"
    else if (grepl("^(swingplus|shape_breakdown)", name)) "swing_plus"
    else "predictiveness"
  d <- file.path(PLOTS, sub)
  dir.create(d, recursive = TRUE, showWarnings = FALSE)
  file.path(d, name)
}

SHAPE     <- c("swing_path_tilt","swing_length","bat_speed","vert_attack_angle","horz_attack_angle_pull")
RAW_SHAPE <- c("swing_path_tilt","swing_length","bat_speed","vert_attack_angle","horz_attack_angle")
cat(sprintf("data dir:  %s\nplots dir: %s\n", DATA, PLOTS))

In [ ]:
swings     <- read_parquet(file.path(DATA, "swings_model.parquet"))
assign     <- read_parquet(file.path(DATA, "cluster_assignments.parquet"))
clust_summ <- read_parquet(file.path(DATA, "cluster_summary.parquet"))
rep        <- read_parquet(file.path(DATA, "batter_repertoire.parquet"))

for (nm in c("swings", "assign", "clust_summ", "rep")) {
  d <- get(nm)
  cat(sprintf("%-12s %d × %d\n", nm, nrow(d), ncol(d)))
}

In [ ]:
# per-swing analysis frame: features + context + value + assigned cluster
swc <- swings |>
  inner_join(assign |> select(play_id, cluster, resp_max), by = "play_id")
cat(sprintf("swc  %d × %d\n", nrow(swc), ncol(swc)))

by_name <- function(s) {
  rep[grepl(s, rep$batter_full_name, ignore.case = TRUE), ]
}

In [ ]:
head(rep)
head(clust_summ)
head(swc)

## Luis Arraez Inspection 

5 Unique Swing Shapes - What Differentiates Each

In [ ]:
ex   <- clust_summ |> filter(grepl("Arraez", batter_full_name, ignore.case = TRUE))
cols <- c("batter_full_name", "cluster", "n", "weight",
          paste0(RAW_SHAPE, "_mean"))
ex |> select(all_of(cols)) |> arrange(cluster) |> print(n = Inf)

## Swing-shape cards (Baseball-Savant style)

| column | what it shows | source metric |
|---|---|---|
| **Attack angle** | vertical direction of the sweet spot at contact (red arrow vs flat) | `vert_attack_angle` |
| **Attack direction** | top-down heading in the hitter's real field orientation — the arrow leans toward their **pull** field (RHH pull = screen-left/LF, LHH pull = screen-right/RF), with the PULL/OPPO labels flipped to match handedness | `horz_attack_angle_pull` (pull frame; screen-flipped for LHH) |
| **Swing tilt** | plane tilt from flat (0°) toward steep | `swing_path_tilt` |
| **Bat speed** | mph vs the MLB-average tick | `bat_speed` |

In [ ]:
REGEN <- FALSE
if (REGEN) {
  system2("/usr/bin/env", args = c("python3", "src/uncommited/swing_cards.py"), wait = TRUE)
}
for (png in c("swing_cards_arraez.png", "swing_cards_judge.png")) {
  p <- plot_path(png)
  if (file.exists(p)) IRdisplay::display_png(file = p)
  else cat(sprintf("Not found: %s\n", p))
}

## Cluster usage by game state (heatmap)

`usage_heatmap(name)` — pass any hitter's name (substring match). Each cell is **P(cluster | game state)**: within a situation, the share of that hitter's swings falling in each shape (columns sum to 100%).

In [ ]:
REGEN <- FALSE
if (REGEN) {
  system2("/usr/bin/env", args = c("python3", "src/uncommited/swing_cards.py", "--heatmap"),
          wait = TRUE)
}
for (png in c("usage_heatmap_arraez.png", "usage_heatmap_judge.png")) {
  p <- plot_path(png)
  if (file.exists(p)) IRdisplay::display_png(file = p)
  else cat(sprintf("Not found: %s\n", p))
}

## Archetype lexicon — eyeballing the vocabulary

The per-batter clusters are bare indices. Layer 1 (`src/swing_label.py`) maps every unit-cluster onto **3 league archetypes**: **Level Oppo / Level Center / Uppercut Pull**. After the handedness fix (pull frame corrected — see below), MLB swing geometry sits on a **level-oppo ↔ uppercut-pull diagonal** (uppercut swings are pull-side), so BIC lands on 3, not a 2×2. `bat_speed` is a **descriptor, not a defining axis**.

Three checks below:
1. **Bat speed vs uppercut** — the correlation motivating the descriptor choice: effort rises with lift, so `bat_speed` is partly redundant with `vert_attack_angle`.
2. **`archetype_map(name)`** — any hitter's shapes on the league grid (x = pull frame, **+ = pull for both hands**); bubble = usage, label = cluster · bat speed.
3. **Collision curve** — a 3-word vocabulary can't uniquely name every shape of a multi-shape hitter, so labels saturate; multi-shape hitters lean on Layer 2 (delta-vs-primary + the situational tag) to separate same-label shapes.

In [ ]:
arch    <- read_parquet(file.path(DATA, "shape_archetypes.parquet"))
lexicon <- read_parquet(file.path(DATA, "archetype_lexicon.parquet"))

ARCH_PAL   <- c("Level Oppo" = "#7EA6D6", "Level Center" = "#d8b483", "Uppercut Pull" = "#e23b3b")
ARCH_ORDER <- intersect(c("Level Oppo", "Level Center", "Uppercut Pull"),
                        unique(arch$archetype_name))
print(lexicon)

## Repertoire size — distribution across the cohort

How many distinct swing shapes each `(batter, stand)` unit carries: post-merge **k** (a hard count) and **effective_shapes** (usage-weighted `exp(entropy)`, discounting rarely-used shapes).

In [ ]:
kc <- rep |>
  count(k, name = "n_units") |>
  arrange(k) |>
  mutate(pct   = n_units / sum(n_units) * 100,
         label = sprintf("%d\n%d%%", n_units, round(pct)))

plt <- ggplot(kc, aes(x = k, y = n_units)) +
  geom_col(fill = "#30a2da", width = 0.7) +
  geom_text(aes(label = label), vjust = -0.3, size = 3.2, lineheight = 0.9) +
  scale_x_continuous(breaks = kc$k) +
  labs(title = "Repertoire size distribution",
       x = "Repertoire size  k  (distinct swing shapes)",
       y = "(batter, stand) units") +
  FT_THEME

out <- plot_path("repertoire_size_distribution.png")
ggsave(out, plt, width = 8, height = 5, dpi = 130)
print(plt)
cat(sprintf("Saved → %s\n", out))